# Phase 3 - Feature engineering

> **📁 Résultat de cette phase :** Le dataset enrichi (features + targets) est sauvegardé dans `data/processed/features_target_600cells.parquet` et `data/processed/features_target_1024cells.parquet`.

Dans cette phase, je transforme les signaux bruts en variables explicatives exploitables pour la modelisation.
L'EDA est deja faite: on garde les patterns observes (court terme, cycle journalier, cycle hebdomadaire) et on construit des features robustes sans fuite d'information.

## Plan de la phase

Dans ce notebook, je vais suivre ce fil logique: (1) charger et verifier les donnees, (2) creer les features temporelles, (3) ajouter les lags et rolling stats, (4) ajouter une feature spatiale, (5) construire la target a 1h, puis (6) nettoyer et sauvegarder le jeu final.

In [2]:
# Imports et options d'affichage
from pathlib import Path
import math
import polars as pl

pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(20)

polars.config.Config

In [3]:
# Chemins d'entree/sortie et verification des dossiers
project_root = Path.cwd()
input_path = project_root / 'data' / 'processed' / 'work_600cells.parquet'
output_dir = project_root / 'data' / 'processed'
output_path = output_dir / 'features_target_600cells.parquet'

if not input_path.exists():
    raise FileNotFoundError(f'Fichier introuvable: {input_path}')

output_dir.mkdir(parents=True, exist_ok=True)
print('Input:', input_path)
print('Output:', output_path)

Input: c:\Users\hp\OneDrive\Desktop\projectTimeSeries\data\processed\work_600cells.parquet
Output: c:\Users\hp\OneDrive\Desktop\projectTimeSeries\data\processed\features_target_600cells.parquet


In [4]:
# Chargement du dataset propre de la phase 1
df = pl.read_parquet(input_path)

print('Dimensions initiales:', df.shape)
print('Nombre de cellules uniques :', df['square_id'].n_unique())
df.head()

Dimensions initiales: (1785594, 5)
Nombre de cellules uniques : 600


square_id,slot_30m,internet_volume,sms_in,call_in
i32,i64,f64,f64,f64
58,1383260400,32.065888,0.496559,0.294724
58,1383262200,30.612743,0.400239,0.298897
58,1383264000,34.056579,0.590001,0.288363
58,1383265800,33.43109,0.226523,0.02857
58,1383267600,24.376322,0.15618,0.116868


## Pourquoi l'encodage cyclique est indispensable

Si je garde l'heure comme un entier brut, le modele voit 47 et 0 comme des valeurs loin l'une de l'autre, alors que ce sont deux instants consecutifs. Le couple sin/cos corrige ce probleme en representant l'heure et le jour sur un cercle continu.

In [5]:
# Tri explicite: indispensable avant les shifts/rolling par cellule
df = df.sort(['square_id', 'slot_30m'])

# Encodage temporel cyclique (continuite entre 23h30 et 00h00)
df = df.with_columns([
    (pl.col('slot_30m') % 86400 // 1800).alias('hour_slot'),
    ((pl.col('slot_30m') // 86400) % 7).alias('dow'),
])

df = df.with_columns([
    (2 * math.pi * pl.col('hour_slot') / 48).sin().alias('sin_hour'),
    (2 * math.pi * pl.col('hour_slot') / 48).cos().alias('cos_hour'),
    (2 * math.pi * pl.col('dow') / 7).sin().alias('sin_dow'),
    (2 * math.pi * pl.col('dow') / 7).cos().alias('cos_dow'),
    (pl.col('dow') >= 5).cast(pl.Int8).alias('is_weekend'),
])

df.select(['hour_slot','dow','sin_hour','cos_hour','sin_dow','cos_dow','is_weekend']).head()

hour_slot,dow,sin_hour,cos_hour,sin_dow,cos_dow,is_weekend
i64,i64,f64,f64,f64,f64,i8
46,0,-0.258819,0.965926,0.0,1.0,0
47,0,-0.130526,0.991445,0.0,1.0,0
0,1,0.0,1.0,0.781831,0.62349,0
1,1,0.130526,0.991445,0.781831,0.62349,0
2,1,0.258819,0.965926,0.781831,0.62349,0


## Intuition des lags

Les lags capturent la memoire de la serie: tres court terme (t-1, t-2), tendance recente (t-6, t-12, t-24), saisonnalite journaliere (t-48, t-96) et saisonnalite hebdomadaire (t-336).

Point important: chaque lag est calcule **par cellule** avec `over('square_id')` pour eviter de melanger des cellules differentes.

In [6]:
# Lags calibres par l'EDA/ACF-PACF: court terme + saisonnalites journaliere et hebdomadaire
lag_steps = [1, 2, 6, 12, 24, 48, 96, 336]

for lag in lag_steps:
    df = df.with_columns(
        pl.col('internet_volume').shift(lag).over('square_id').alias(f'lag_{lag}')
    )

print('Lags ajoutes:', [f'lag_{lag}' for lag in lag_steps])
df.select(['square_id','slot_30m','internet_volume','lag_1','lag_48','lag_336']).head(10)

Lags ajoutes: ['lag_1', 'lag_2', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'lag_96', 'lag_336']


square_id,slot_30m,internet_volume,lag_1,lag_48,lag_336
i32,i64,f64,f64,f64,f64
58,1383260400,32.065888,null,null,null
58,1383262200,30.612743,32.065888,null,null
58,1383264000,34.056579,30.612743,null,null
58,1383265800,33.43109,34.056579,null,null
…,…,…,…,…,…
58,1383271200,21.733877,21.583401,null,null
58,1383273000,18.630972,21.733877,null,null
58,1383274800,20.429251,18.630972,null,null
58,1383276600,17.561255,20.429251,null,null


## Intuition des rolling statistics

La moyenne, l'ecart-type et le max glissants donnent une vue plus stable du contexte local (niveau moyen, volatilite, pics).

J'applique un `shift(1)` avant les fenetres pour garantir que les features utilisent uniquement le passe.

In [7]:
# Rolling statistics sur 3h, 6h, 24h
# Important: shift(1) pour n'utiliser que le passe (pas de fuite d'information).
windows = {'3h': 6, '6h': 12, '24h': 48}

for name, w in windows.items():
    shifted_series = pl.col('internet_volume').shift(1)
    df = df.with_columns([
        shifted_series.rolling_mean(w).over('square_id').alias(f'roll_mean_{name}'),
        shifted_series.rolling_std(w).over('square_id').alias(f'roll_std_{name}'),
        shifted_series.rolling_max(w).over('square_id').alias(f'roll_max_{name}'),
    ])

roll_cols = ['roll_mean_3h', 'roll_std_3h', 'roll_max_3h', 'roll_mean_24h', 'roll_std_24h', 'roll_max_24h']
df.select(['square_id', 'slot_30m', 'internet_volume', *roll_cols]).head(10)

square_id,slot_30m,internet_volume,roll_mean_3h,roll_std_3h,roll_max_3h,roll_mean_24h,roll_std_24h,roll_max_24h
i32,i64,f64,f64,f64,f64,f64,f64,f64
58,1383260400,32.065888,null,null,null,null,null,null
58,1383262200,30.612743,null,null,null,null,null,null
58,1383264000,34.056579,null,null,null,null,null,null
58,1383265800,33.43109,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…
58,1383271200,21.733877,29.354337,5.154322,34.056579,null,null,null
58,1383273000,18.630972,27.632335,5.757818,34.056579,null,null,null
58,1383274800,20.429251,25.635373,6.541818,34.056579,null,null,null
58,1383276600,17.561255,23.364152,5.276613,33.43109,null,null,null


## Feature spatiale: ce qu'elle apporte

Le trafic d'une cellule est souvent influence par ses voisines. Je calcule donc la moyenne des 8 voisins (Moore), puis je decale de 2 pas pour rester causal (pas de leakage).

Cette feature est utile pour capter une propagation locale de congestion.

In [8]:
# Feature spatiale: moyenne des 8 voisins (Moore) au pas t-2 pour eviter la fuite d'information
def get_moore_neighbors(cell_id: int, grid_size: int = 100) -> list[int]:
    row, col = (cell_id - 1) // grid_size, (cell_id - 1) % grid_size
    neighbors = []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue
            r, c = row + dr, col + dc
            if 0 <= r < grid_size and 0 <= c < grid_size:
                neighbors.append(r * grid_size + c + 1)
    return neighbors

present_cells = set(df['square_id'].unique().to_list())
neighbor_map = {cid: [n for n in get_moore_neighbors(cid) if n in present_cells] for cid in present_cells}

timestamps = df['slot_30m'].unique().sort().to_list()
spatial_frames = []

for ts in timestamps:
    frame_t = df.filter(pl.col('slot_30m') == ts).select(['square_id', 'internet_volume'])
    volume_map = dict(zip(frame_t['square_id'].to_list(), frame_t['internet_volume'].to_list()))

    rows = []
    for cid in frame_t['square_id'].to_list():
        vals = [volume_map[n] for n in neighbor_map[cid] if n in volume_map]
        neigh_mean = float(sum(vals) / len(vals)) if vals else None
        rows.append((cid, ts, neigh_mean))

    spatial_frames.append(pl.DataFrame(rows, schema=['square_id', 'slot_30m', 'neighbor_mean_t0'], orient='row'))

spatial_df = pl.concat(spatial_frames) if spatial_frames else pl.DataFrame(schema={'square_id': pl.Int64, 'slot_30m': pl.Int64, 'neighbor_mean_t0': pl.Float64})
spatial_df = spatial_df.with_columns(
    pl.col('neighbor_mean_t0').shift(2).over('square_id').alias('neighbor_mean_t_minus_2')
)

df = df.join(
    spatial_df.select(['square_id', 'slot_30m', 'neighbor_mean_t_minus_2']),
    on=['square_id', 'slot_30m'],
    how='left',
)

df.select(['square_id','slot_30m','neighbor_mean_t_minus_2']).head(10)

square_id,slot_30m,neighbor_mean_t_minus_2
i32,i64,f64
58,1383260400,null
58,1383262200,null
58,1383264000,null
58,1383265800,null
…,…,…
58,1383271200,null
58,1383273000,null
58,1383274800,null
58,1383276600,null


## Choix de la target

La cible est `target_1h = y(t+2)` car 2 slots de 30 minutes = 1h.

Cet horizon est un bon compromis: assez proche pour rester predictible, mais assez long pour laisser un temps d'action operationnel.

In [9]:
# Target a 1h: 2 pas de 30 min dans le futur
df = df.with_columns(
    pl.col('internet_volume').shift(-2).over('square_id').alias('target_1h')
)

df.select(['square_id','slot_30m','internet_volume','target_1h']).head(10)

square_id,slot_30m,internet_volume,target_1h
i32,i64,f64,f64
58,1383260400,32.065888,34.056579
58,1383262200,30.612743,33.43109
58,1383264000,34.056579,24.376322
58,1383265800,33.43109,21.583401
…,…,…,…
58,1383271200,21.733877,20.429251
58,1383273000,18.630972,17.561255
58,1383274800,20.429251,19.287903
58,1383276600,17.561255,20.622869


## Nettoyage et impact attendu

Le nettoyage supprime surtout le debut de chaque serie (a cause du lag 336) et les 2 derniers points (target future).

Cette perte de lignes est normale et necessaire pour obtenir un dataset strictement exploitable.

In [10]:
# Nettoyage final: retirer debut (lags/rolling) et fin (target future)
rows_before = df.height

required_cols = ['target_1h', *[f'lag_{lag}' for lag in lag_steps], 'neighbor_mean_t_minus_2']
df_final = df.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in required_cols]))
rows_after = df_final.height

print(f'Lignes avant nettoyage: {rows_before:,}')
print(f'Lignes apres nettoyage: {rows_after:,}')
print(f'Lignes supprimees: {rows_before - rows_after:,}')

Lignes avant nettoyage: 1,785,594
Lignes apres nettoyage: 1,316,362
Lignes supprimees: 469,232


In [11]:
# Sauvegarde parquet (compression zstd) pour la phase 4
df_final.write_parquet(output_path, compression='zstd')
print('Fichier sauvegarde:', output_path)

Fichier sauvegarde: c:\Users\hp\OneDrive\Desktop\projectTimeSeries\data\processed\features_target_600cells.parquet


## Lecture rapide des resultats

Si `NaN/null = 0`, le dataset est propre pour la phase 4.
Le nombre de colonnes doit refleter toutes les familles de features (temporelles, lags, rolling, spatiale) + la target.

In [12]:
# Verification finale: apercu, colonnes, et controle des valeurs manquantes
print('Dimensions finales:', df_final.shape)
print('Nombre total de NaN/null:', int(df_final.null_count().select(pl.sum_horizontal(pl.all())).item()))

print('\nColonnes finales:')
print(df_final.columns)

df_final.head()

Dimensions finales: (1316362, 31)
Nombre total de NaN/null: 0

Colonnes finales:
['square_id', 'slot_30m', 'internet_volume', 'sms_in', 'call_in', 'hour_slot', 'dow', 'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'is_weekend', 'lag_1', 'lag_2', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'lag_96', 'lag_336', 'roll_mean_3h', 'roll_std_3h', 'roll_max_3h', 'roll_mean_6h', 'roll_std_6h', 'roll_max_6h', 'roll_mean_24h', 'roll_std_24h', 'roll_max_24h', 'neighbor_mean_t_minus_2', 'target_1h']


square_id,slot_30m,internet_volume,sms_in,call_in,hour_slot,dow,sin_hour,cos_hour,sin_dow,…,roll_std_3h,roll_max_3h,roll_mean_6h,roll_std_6h,roll_max_6h,roll_mean_24h,roll_std_24h,roll_max_24h,neighbor_mean_t_minus_2,target_1h
i32,i64,f64,f64,f64,i64,i64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3756,1383865200,1306.934109,24.730153,12.043053,46,0,-0.258819,0.965926,0.0,…,106.693811,1415.009148,1156.079633,129.67853,1415.009148,868.223051,221.565805,1415.009148,1344.487493,1141.257046
3756,1383867000,1108.578012,16.502316,4.402276,47,0,-0.130526,0.991445,0.0,…,108.848068,1415.009148,1180.424351,128.156928,1415.009148,873.48718,228.958099,1415.009148,1247.145565,1029.775877
3756,1383868800,1141.257046,7.083825,2.629166,0,1,0.0,1.0,0.781831,…,124.967378,1415.009148,1185.941191,122.995005,1415.009148,874.76936,230.120025,1415.009148,1228.10927,790.6688
3756,1383870600,1029.775877,5.199155,1.501319,1,1,0.130526,0.991445,0.781831,…,121.796616,1415.009148,1189.933809,120.467621,1415.009148,877.677921,232.626425,1415.009148,1074.510722,778.299742
3756,1383872400,790.6688,7.961658,0.953884,2,1,0.258819,0.965926,0.781831,…,91.944515,1306.934109,1187.080789,124.072391,1415.009148,880.345197,233.640045,1415.009148,1063.872849,732.802194
